# Super-Resolution Evaluation
This notebook evaluates SR models across scales with consistent splits, metrics, and rich visualizations.

In [ ]:
pip install seaborn

In [ ]:

# Setup
import os
import json
from pathlib import Path
import math
import random

import numpy as np
import torch
import torch.utils.data
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm

import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAVE_SNS = True
except Exception:
    _HAVE_SNS = False

# Local imports
from dataloader import SuperResDataset, split_indices
from dataloader.utils import list_images
from models.superresolution import SRCNN, AutoencoderSR, SRGANGenerator, SwinIR
from utils.superresolution import psnr, ssim, rgb_to_y, match_size

# Device
if torch.cuda.is_available():
    device = torch.device("cuda:1")
else:
    device = torch.device("cpu")
print("Using device:", device)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:

# Paths & runs
HR_ROOT = Path("/home/data/SyMTRS/hr")
LR_ROOTS = {
    2: Path("/home/data/SyMTRS/lr/x2"),
    4: Path("/home/data/SyMTRS/lr/x4"),
    8: Path("/home/data/SyMTRS/lr/x8"),
}

RUNS = {
    2: {
        "autoencoder": Path("sr_runs_autoencoder_x2"),
        "srcnn": Path("sr_runs_srcnn_x2"),
        "srgan": Path("sr_runs_srgan_x2"),
        "swinir": Path("sr_runs_swinir_x2"),
    },
    4: {
        "autoencoder": Path("sr_runs_autoencoder_x4"),
        "srcnn": Path("sr_runs_srcnn_x4"),
        "srgan": Path("sr_runs_srgan_x4"),
        "swinir": Path("sr_runs_swinir_x4"),
    },
    8: {
        "autoencoder": Path("sr_runs_autoencoder_x8"),
        "srcnn": Path("sr_runs_srcnn_x8"),
        "srgan": Path("sr_runs_srgan_x8"),
        "swinir": Path("sr_runs_swinir_x8"),
    },
}

# Split ratios and evaluation params
SPLIT_RATIOS = (0.8, 0.2)  # train/test
BATCH_SIZE = 1
NUM_WORKERS = 4
N_VIS = 4  # number of samples to visualize per scale

OUT_DIR = Path("sr_eval_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

# Helpers

def collate_same_size(batch):
    if not batch:
        return None
    ref = batch[0]["lr"].shape
    filtered = [b for b in batch if b["lr"].shape == ref and b["hr"].shape == batch[0]["hr"].shape]
    if len(filtered) < len(batch):
        skipped = len(batch) - len(filtered)
        print(f"[collate] Skipping {skipped} sample(s) due to mismatched sizes in batch")
    if not filtered:
        return None
    return torch.utils.data.default_collate(filtered)


def prepare_batch(lr, hr, model_name, use_y):
    if use_y:
        lr = rgb_to_y(lr)
        hr = rgb_to_y(hr)
    if model_name in {"srcnn", "autoencoder"}:
        lr = F.interpolate(lr, size=hr.shape[-2:], mode="bicubic", align_corners=False)
    return lr, hr


def strip_module_prefix(state_dict):
    if not any(k.startswith("module.") for k in state_dict.keys()):
        return state_dict
    return {k.replace("module.", "", 1): v for k, v in state_dict.items()}


def build_model_from_args(args, img_size=None):
    model_name = args.get("model")
    scale = int(args.get("scale", 4))
    use_y = bool(args.get("use_y_channel", False))
    in_ch = 1 if use_y else 3
    out_ch = in_ch

    if model_name == "srcnn":
        return SRCNN(in_channels=in_ch, out_channels=out_ch)
    if model_name == "autoencoder":
        return AutoencoderSR(in_channels=in_ch)
    if model_name == "srgan":
        return SRGANGenerator(in_channels=in_ch, out_channels=out_ch, scale=scale)
    if model_name == "swinir":
        if img_size is None:
            raise ValueError("img_size required for SwinIR")
        return SwinIR(
            img_size=img_size,
            in_chans=in_ch,
            embed_dim=int(args.get("swinir_embed_dim", 96)),
            depths=args.get("swinir_depths", [6,6,6,6]),
            num_heads=args.get("swinir_num_heads", [6,6,6,6]),
            window_size=int(args.get("swinir_window", 7)),
            mlp_ratio=float(args.get("swinir_mlp_ratio", 4.0)),
            upscale=scale,
            img_range=1.0,
            upsampler=args.get("swinir_upsampler", "pixelshuffle"),
            use_checkpoint=bool(args.get("swinir_checkpoint", False)),
        )
    raise ValueError(f"Unknown model: {model_name}")


def load_model_from_run(run_dir, sample_lr_shape=None):
    run_dir = Path(run_dir)
    ckpt_path = run_dir / "weights" / "best.pt"
    if not ckpt_path.exists():
        ckpt_path = run_dir / "weights" / "last.pt"
    if not ckpt_path.exists():
        raise FileNotFoundError(f"No checkpoint found in {run_dir}")

    ckpt = torch.load(ckpt_path, map_location="cpu")
    args = ckpt.get("args", {})

    img_size = None
    if args.get("model") == "swinir":
        if sample_lr_shape is None:
            raise ValueError("sample_lr_shape required for SwinIR")
        h, w = sample_lr_shape[-2:]
        ws = int(args.get("swinir_window", 7))
        h = ((h + ws - 1) // ws) * ws
        w = ((w + ws - 1) // ws) * ws
        img_size = (h, w)

    model = build_model_from_args(args, img_size=img_size)
    model.load_state_dict(strip_module_prefix(ckpt["model"]))
    model.to(device)
    model.eval()
    return model, args


def make_split_stems(hr_root, ratios=(0.8,0.2), seed=42):
    hr_files = list_images(hr_root)
    stems = [p.stem for p in hr_files]
    splits = split_indices(len(stems), ratios=ratios, seed=seed, shuffle=True)
    split_stems = {
        "train": [stems[i] for i in splits[0]],
        "test": [stems[i] for i in splits[1]],
    }
    return split_stems


def stems_to_indices(dataset, stems):
    # dataset.pairs is list of (lr_path, hr_path)
    index_map = {Path(hr).stem: i for i, (_, hr) in enumerate(dataset.pairs)}
    idxs = [index_map[s] for s in stems if s in index_map]
    missing = [s for s in stems if s not in index_map]
    if missing:
        print(f"[warn] Missing {len(missing)} stems in dataset")
    return idxs


def tensor_to_img(t):
    # t: CxHxW in [0,1]
    if t.ndim == 3 and t.shape[0] == 1:
        t = t.repeat(3, 1, 1)
    t = t.detach().cpu().clamp(0,1)
    return (t.permute(1,2,0).numpy() * 255.0).astype(np.uint8)


def compute_metrics_batch(sr, hr):
    # returns per-sample metrics
    b = sr.shape[0]
    mse_vals = []
    psnr_vals = []
    ssim_vals = []
    for i in range(b):
        s = sr[i:i+1]
        h = hr[i:i+1]
        mse = F.mse_loss(s, h).item()
        mse_vals.append(mse)
        psnr_vals.append(psnr(s, h).item())
        ssim_vals.append(ssim(s, h).item())
    return mse_vals, psnr_vals, ssim_vals


def radar_plot(ax, labels, values_dict, title=None):
    # values_dict: {name: [vals...]}
    angles = np.linspace(0, 2*np.pi, len(labels), endpoint=False).tolist()
    angles += angles[:1]
    for name, vals in values_dict.items():
        v = vals + vals[:1]
        ax.plot(angles, v, label=name)
        ax.fill(angles, v, alpha=0.15)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels)
    ax.set_yticklabels([])
    if title:
        ax.set_title(title)
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))


In [ ]:

# Create a consistent split (shared across all scales/models)
split_stems = make_split_stems(HR_ROOT, ratios=SPLIT_RATIOS, seed=SEED)

split_path = OUT_DIR / "sr_eval_split.json"
with open(split_path, "w") as f:
    json.dump({"ratios": SPLIT_RATIOS, "seed": SEED, "split_stems": split_stems}, f, indent=2)

print("Split saved to", split_path)
print({k: len(v) for k, v in split_stems.items()})


In [ ]:

# Evaluate all models
import pandas as pd

all_results = []
per_image_results = []

for scale, runs in RUNS.items():
    print(f"=== Scale x{scale} ===")
    lr_root = LR_ROOTS[scale]
    dataset = SuperResDataset(lr_root, HR_ROOT)
    test_indices = stems_to_indices(dataset, split_stems["test"])
    test_ds = Subset(dataset, test_indices)
    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_same_size,
    )

    # sample for swinir init
    sample_lr = dataset[0]["lr"]

    for model_name, run_dir in runs.items():
        print(f"Evaluating {model_name} ({run_dir})")
        model, args = load_model_from_run(run_dir, sample_lr_shape=sample_lr.shape)
        use_y = bool(args.get("use_y_channel", False))
        model_tag = f"{model_name}_x{scale}"

        mse_vals = []
        psnr_vals = []
        ssim_vals = []

        with torch.no_grad():
            for batch in tqdm(test_loader, desc=f"{model_tag}", leave=False):
                if batch is None:
                    continue
                lr = batch["lr"].to(device)
                hr = batch["hr"].to(device)
                lr, hr = prepare_batch(lr, hr, model_name, use_y)

                sr = model(lr)
                hr_c, sr_c = match_size(hr, sr)

                m, p, s = compute_metrics_batch(sr_c, hr_c)
                mse_vals += m
                psnr_vals += p
                ssim_vals += s

        all_results.append({
            "scale": scale,
            "model": model_name,
            "mse": float(np.mean(mse_vals)),
            "psnr": float(np.mean(psnr_vals)),
            "ssim": float(np.mean(ssim_vals)),
            "n": len(psnr_vals),
        })

        for i, (m, p, s) in enumerate(zip(mse_vals, psnr_vals, ssim_vals)):
            per_image_results.append({
                "scale": scale,
                "model": model_name,
                "mse": m,
                "psnr": p,
                "ssim": s,
            })


        # cleanup model to free GPU memory
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
results_df = pd.DataFrame(all_results)
per_img_df = pd.DataFrame(per_image_results)

results_df


In [ ]:

# Aggregate metrics tables
for scale in sorted(results_df["scale"].unique()):
    print(f"Scale x{scale}")
    display(results_df[results_df["scale"] == scale].sort_values("psnr", ascending=False))


In [ ]:
# Plot: bar charts (single figure with PSNR/SSIM/MSE), bars = x2/x4/x8 per model
scales = sorted(results_df["scale"].unique())
metrics = ["psnr", "ssim", "mse"]
metric_titles = {"psnr": "PSNR", "ssim": "SSIM", "mse": "MSE"}
scale_colors = {2: "#1f77b4", 4: "#ff7f0e", 8: "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
if len(metrics) == 1:
    axes = [axes]

models = results_df["model"].unique().tolist()

for ax, metric in zip(axes, metrics):
    x = np.arange(len(models))
    width = 0.22

    # best value per metric (max for PSNR/SSIM, min for MSE)
    best_val = results_df[metric].min() if metric == "mse" else results_df[metric].max()

    for i, scale in enumerate(scales):
        df = results_df[results_df["scale"] == scale].set_index("model")
        vals = [df.loc[m, metric] for m in models]
        offsets = x + (i - (len(scales)-1)/2) * width
        for j, v in enumerate(vals):
            is_best = v == best_val
            ax.bar(
                offsets[j], v, width=width,
                color=scale_colors.get(scale, None),
                edgecolor="black" if is_best else "none",
                linewidth=2.2 if is_best else 0.0,
            )

    ax.set_title(metric_titles[metric])
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=30)
    ax.set_ylabel("value")

# legend for scales
handles = [
    plt.Line2D([0], [0], color=scale_colors[s], lw=6, label=f"x{s}")
    for s in scales
]
fig.legend(handles=handles, loc="upper center", ncol=len(scales), bbox_to_anchor=(0.5, 1.05))
plt.tight_layout()
out = OUT_DIR / "bar_metrics_by_metric.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# Plot: radar/spider plot per scale (normalized)
for scale in sorted(results_df["scale"].unique()):
    df = results_df[results_df["scale"] == scale].copy()
    # Normalize metrics to [0,1] for radar: higher is better. For MSE invert.
    df["mse_inv"] = 1.0 / (df["mse"] + 1e-8)
    for col in ["psnr", "ssim", "mse_inv"]:
        v = df[col].values
        df[col] = (v - v.min()) / (v.max() - v.min() + 1e-8)

    fig = plt.figure(figsize=(5, 5))
    ax = plt.subplot(111, polar=True)
    values_dict = {row["model"]: [row["psnr"], row["ssim"], row["mse_inv"]] for _, row in df.iterrows()}
    radar_plot(ax, ["PSNR", "SSIM", "1/MSE"], values_dict, title=f"Radar x{scale}")
    out = OUT_DIR / f"radar_x{scale}.png"
    plt.savefig(out, dpi=200, bbox_inches="tight")
    plt.show()


In [ ]:
# Plot: per-image metric distributions (single figure with PSNR/SSIM/MSE)
scales = sorted(per_img_df["scale"].unique())
metrics = ["psnr", "ssim", "mse"]
metric_titles = {"psnr": "PSNR", "ssim": "SSIM", "mse": "MSE"}
scale_colors = {2: "#1f77b4", 4: "#ff7f0e", 8: "#2ca02c"}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
if len(metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, metrics):
    df = per_img_df[["scale", "model", metric]].copy()
    df = df.rename(columns={metric: "value"})

    if _HAVE_SNS:
        sns.boxplot(
            data=df, x="model", y="value", hue="scale",
            palette=[scale_colors[s] for s in scales], ax=ax
        )
    else:
        # fallback: grouped boxplots per model with scale offsets
        models = sorted(df["model"].unique())
        width = 0.22
        positions = []
        data = []
        for i, scale in enumerate(scales):
            for j, model in enumerate(models):
                vals = df[(df["scale"] == scale) & (df["model"] == model)]["value"].values
                data.append(vals)
                positions.append(j + (i - (len(scales)-1)/2) * width)
        bp = ax.boxplot(data, positions=positions, widths=width, patch_artist=True)
        # color boxes by scale
        for k, box in enumerate(bp["boxes"]):
            scale_idx = k // len(models)
            scale = scales[scale_idx]
            box.set_facecolor(scale_colors.get(scale, "#777777"))

        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=30)

    # highlight best median (max for PSNR/SSIM, min for MSE)
    medians = df.groupby(["scale", "model"])['value'].median().reset_index()
    if metric == "mse":
        best_row = medians.loc[medians['value'].idxmin()]
    else:
        best_row = medians.loc[medians['value'].idxmax()]
    best_model = best_row['model']
    best_scale = best_row['scale']
    best_val = best_row['value']

    # compute x-position for marker
    models = results_df["model"].unique().tolist()
    m_idx = models.index(best_model)
    s_idx = scales.index(best_scale)
    x_pos = m_idx + (s_idx - (len(scales)-1)/2) * 0.22
    ax.scatter([x_pos], [best_val], s=80, marker='*', color='black', zorder=5)

    ax.set_title(metric_titles[metric])
    ax.set_ylabel("value")

# legend for scales
handles = [
    plt.Line2D([0], [0], color=scale_colors[s], lw=6, label=f"x{s}")
    for s in scales
]
fig.legend(handles=handles, loc="upper center", ncol=len(scales), bbox_to_anchor=(0.5, 1.05))
plt.tight_layout()
out = OUT_DIR / "distributions_by_metric.png"
plt.savefig(out, dpi=200, bbox_inches="tight")
plt.show()


In [ ]:

# Plot: PSNR vs SSIM scatter
for scale in sorted(per_img_df["scale"].unique()):
    df = per_img_df[per_img_df["scale"] == scale]
    plt.figure(figsize=(5, 4))
    if _HAVE_SNS:
        sns.scatterplot(data=df, x="psnr", y="ssim", hue="model", alpha=0.6)
    else:
        for m in df["model"].unique():
            d = df[df["model"] == m]
            plt.scatter(d["psnr"], d["ssim"], label=m, alpha=0.6)
        plt.legend()
    plt.title(f"PSNR vs SSIM (x{scale})")
    plt.tight_layout()
    out = OUT_DIR / f"scatter_psnr_ssim_x{scale}.png"
    plt.savefig(out, dpi=200)
    plt.show()


In [ ]:

# Visualize predictions: LR / SR / HR / Error

def visualize_samples(scale, n_samples=4):
    lr_root = LR_ROOTS[scale]
    dataset = SuperResDataset(lr_root, HR_ROOT)
    test_indices = stems_to_indices(dataset, split_stems["test"])
    test_ds = Subset(dataset, test_indices)

    save_dir = OUT_DIR / f"preds_x{scale}"
    save_dir.mkdir(parents=True, exist_ok=True)

    # pick samples
    idxs = list(range(min(n_samples, len(test_ds))))
    samples = [test_ds[i] for i in idxs]

    # pre-load models
    models = {}
    sample_lr = dataset[0]["lr"]
    for model_name, run_dir in RUNS[scale].items():
        model, args = load_model_from_run(run_dir, sample_lr_shape=sample_lr.shape)
        models[model_name] = (model, args)

    for i, sample in enumerate(samples):
        lr = sample["lr"].unsqueeze(0).to(device)
        hr = sample["hr"].unsqueeze(0).to(device)

        fig, axes = plt.subplots(len(models), 4, figsize=(12, 3*len(models)))
        if len(models) == 1:
            axes = np.expand_dims(axes, 0)

        for row, (model_name, (model, args)) in enumerate(models.items()):
            use_y = bool(args.get("use_y_channel", False))
            lr_p, hr_p = prepare_batch(lr, hr, model_name, use_y)
            with torch.no_grad():
                sr = model(lr_p)
            hr_c, sr_c = match_size(hr_p, sr)
            lr_c, _ = match_size(lr_p, sr_c)

            # error map (absolute diff)
            err = (hr_c - sr_c).abs().mean(dim=1, keepdim=True)

            # save individual images for custom canvas
            stem = Path(sample["hr_path"]).stem if "hr_path" in sample else f"sample{i}"
            base = save_dir / f"{stem}_{model_name}"
            plt.imsave(str(base) + "_lr.png", tensor_to_img(lr_c[0]))
            plt.imsave(str(base) + "_sr.png", tensor_to_img(sr_c[0]))
            plt.imsave(str(base) + "_hr.png", tensor_to_img(hr_c[0]))
            plt.imsave(str(base) + "_err.png", tensor_to_img(err[0]), cmap="magma")

            axes[row, 0].imshow(tensor_to_img(lr_c[0]))
            axes[row, 0].set_title(f"{model_name} LR")
            axes[row, 1].imshow(tensor_to_img(sr_c[0]))
            axes[row, 1].set_title(f"{model_name} SR")
            axes[row, 2].imshow(tensor_to_img(hr_c[0]))
            axes[row, 2].set_title(f"{model_name} HR")
            axes[row, 3].imshow(tensor_to_img(err[0]), cmap="magma")
            axes[row, 3].set_title(f"{model_name} |HR-SR|")

            for c in range(4):
                axes[row, c].axis("off")

            # cleanup per-model tensors
            del sr, hr_c, sr_c, lr_c
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        plt.tight_layout()
        out = OUT_DIR / f"vis_x{scale}_sample{i}.png"
        plt.savefig(out, dpi=200)
        plt.show()

        # cleanup models to free GPU memory
        for _m, _a in models.values():
            del _m
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for scale in [2, 4, 8]:
    visualize_samples(scale, n_samples=N_VIS)


In [ ]:

# Plot training curves from each run (if metrics.csv exists)
import pandas as pd

METRIC_COLUMNS = [
    "epoch",
    "train_loss",
    "val_loss",
    "train_psnr",
    "val_psnr",
    "mse_train",
    "mse_val",
    "ssim_train",
    "ssim_val",
]

for scale, runs in RUNS.items():
    for model_name, run_dir in runs.items():
        csv_path = Path(run_dir) / "metrics.csv"
        if not csv_path.exists():
            continue

        # Robust read: supports files mixing 5-col and 9-col rows.
        df = pd.read_csv(
            csv_path,
            header=None,
            names=METRIC_COLUMNS,
            skiprows=1,
            engine="python",
            on_bad_lines="skip",
        )
        for col in METRIC_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.dropna(
            subset=["epoch", "train_loss", "val_loss", "train_psnr", "val_psnr"]
        ).sort_values("epoch")
        if df.empty:
            continue

        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
        axes[0].plot(df["epoch"], df["train_loss"], label="train")
        axes[0].plot(df["epoch"], df["val_loss"], label="val")
        axes[0].set_title(f"Loss {model_name} x{scale}")
        axes[1].plot(df["epoch"], df["train_psnr"], label="train")
        axes[1].plot(df["epoch"], df["val_psnr"], label="val")
        axes[1].set_title(f"PSNR {model_name} x{scale}")

        has_ssim = df[["ssim_train", "ssim_val"]].notna().any().any()
        if has_ssim:
            axes[2].plot(df["epoch"], df["ssim_train"], label="train")
            axes[2].plot(df["epoch"], df["ssim_val"], label="val")
            axes[2].set_title(f"SSIM {model_name} x{scale}")
        else:
            axes[2].axis("off")

        for ax in axes:
            if ax.has_data():
                ax.legend()
        plt.tight_layout()
        out = OUT_DIR / f"curves_{model_name}_x{scale}.png"
        plt.savefig(out, dpi=200)
        plt.show()
